# SignalScope Dataset Collection and EDA

This notebook builds the first dataset for **SignalScope** and performs exploratory data analysis before the Q-learning environment is created.

The project uses four signal families:

1. **Market data** from Yahoo Finance  
2. **Retail attention data** from Google Trends  
3. **Official company disclosure data** from SEC EDGAR  
4. **Optional news data** from Finnhub  

The goal of this notebook is not to train the RL agent yet.  
The goal is to create a clean analytical dataset and check whether the signals have any relationship with short-term returns, volume, volatility, and drawdowns.

## 0. Installation

Run this only once in your environment.

```bash
pip install pandas numpy matplotlib yfinance pytrends sec-edgar-downloader beautifulsoup4 finnhub-python tqdm
```

In [1]:
from pathlib import Path
import os
import re
import json
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Optional imports. The notebook will tell you clearly if something is missing.
try:
    import yfinance as yf
except Exception as e:
    yf = None
    print("yfinance not available:", e)

try:
    from pytrends.request import TrendReq
except Exception as e:
    TrendReq = None
    print("pytrends not available:", e)

try:
    from sec_edgar_downloader import Downloader
except Exception as e:
    Downloader = None
    print("sec_edgar_downloader not available:", e)

try:
    from bs4 import BeautifulSoup
except Exception as e:
    BeautifulSoup = None
    print("BeautifulSoup not available:", e)

try:
    import finnhub
except Exception as e:
    finnhub = None
    print("finnhub-python not available:", e)

In [ ]:
# -------------------------
# Project configuration
# -------------------------

PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

YFINANCE_DIR = RAW_DIR / "yfinance"
TRENDS_DIR = RAW_DIR / "google_trends"
SEC_DIR = Path("data/sec")
FINNHUB_DIR = RAW_DIR / "finnhub"
SEC_FILINGS_DIR = SEC_DIR / "sec-edgar-filings"


for p in [DATA_DIR, RAW_DIR, PROCESSED_DIR, YFINANCE_DIR, TRENDS_DIR, SEC_DIR, FINNHUB_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TICKERS = ["NVDA", "MSFT", "GOOG", "META", "AMZN", "AAPL"]

START_DATE = "2022-01-01"
END_DATE = "2025-01-01"
GEO = "US"

AI_TREND_KEYWORDS = [
    "artificial intelligence",
    "ChatGPT",
    "Claude AI",
    "generative AI",
    "machine learning",
]

SEC_AI_KEYWORDS = [
    "artificial intelligence",
    "generative ai",
    "machine learning",
    "large language model",
    "llm",
    "foundation model",
    "data center",
    "accelerated computing",
    "automation",
]

print("Project root:", PROJECT_ROOT)

Project root: e:\Om\signalscope


## 1. Helper Functions

These functions do four things:

1. Download and cache datasets  
2. Clean raw data  
3. Create features  
4. Build one merged panel dataset by `date` and `ticker`  

The notebook always saves raw files first so that EDA does not repeatedly hit APIs.

In [3]:
def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Saved: {path}")

def read_csv_if_exists(path: Path) -> pd.DataFrame | None:
    if path.exists():
        print(f"Loading cached file: {path}")
        return pd.read_csv(path)
    return None

def flatten_yfinance_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
    return df

def add_market_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.sort_values(["ticker", "date"])

    df["daily_return"] = df.groupby("ticker")["close"].pct_change()
    df["log_return"] = np.log(df["close"] / df.groupby("ticker")["close"].shift(1))

    df["return_5d_fwd"] = df.groupby("ticker")["close"].shift(-5) / df["close"] - 1
    df["return_10d_fwd"] = df.groupby("ticker")["close"].shift(-10) / df["close"] - 1

    df["volatility_10d"] = (
        df.groupby("ticker")["daily_return"]
        .rolling(10)
        .std()
        .reset_index(level=0, drop=True)
    )

    df["volume_mean_20d"] = (
        df.groupby("ticker")["volume"]
        .rolling(20)
        .mean()
        .reset_index(level=0, drop=True)
    )

    df["volume_std_20d"] = (
        df.groupby("ticker")["volume"]
        .rolling(20)
        .std()
        .reset_index(level=0, drop=True)
    )

    df["volume_zscore"] = (df["volume"] - df["volume_mean_20d"]) / df["volume_std_20d"]

    # Forward 5-day drawdown from current close
    def forward_drawdown(close: pd.Series, window: int = 5) -> pd.Series:
        values = close.to_numpy()
        out = np.full(len(values), np.nan)
        for i in range(len(values)):
            future = values[i + 1:i + window + 1]
            if len(future) > 0 and values[i] != 0:
                out[i] = future.min() / values[i] - 1
        return pd.Series(out, index=close.index)

    df["drawdown_5d_fwd"] = (
        df.groupby("ticker")["close"]
        .apply(lambda x: forward_drawdown(x, 5))
        .reset_index(level=0, drop=True)
    )

    return df

def bucket_series(s: pd.Series, labels=("low", "medium", "high")) -> pd.Series:
    # qcut fails when the series has too many duplicate values, so fall back to cut.
    try:
        return pd.qcut(s, q=len(labels), labels=labels, duplicates="drop")
    except Exception:
        return pd.cut(s, bins=len(labels), labels=labels)

## 2. Collect Stock Price and Volume Data

This creates the market backbone of the dataset.

Features created from this source:

- Daily return
- 5-day forward return
- 10-day forward return
- 10-day volatility
- 20-day volume z-score
- 5-day forward drawdown

In [12]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

YFINANCE_DIR = RAW_DIR / "yfinance"
YFINANCE_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def save_csv(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def read_csv_if_exists(path):
    if Path(path).exists():
        return pd.read_csv(path)
    return None


def normalize_yfinance_frame(df, ticker):
    df = df.copy()

    # Flatten MultiIndex columns if yfinance returns them
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            "_".join([str(x) for x in col if str(x) != ""])
            for col in df.columns
        ]

    # Bring index into columns if date is stored as index
    df = df.reset_index()

    # Clean column names
    df.columns = [
        str(c)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace(".", "_")
        for c in df.columns
    ]

    # yfinance sometimes creates columns like close_nvda, volume_nvda
    rename_map = {}
    for col in df.columns:
        base = col.replace(f"_{ticker.lower()}", "")
        if base in ["date", "open", "high", "low", "close", "adj_close", "volume"]:
            rename_map[col] = base

    df = df.rename(columns=rename_map)

    # Detect date column
    possible_date_cols = ["date", "datetime", "index"]
    date_col = None

    for col in possible_date_cols:
        if col in df.columns:
            date_col = col
            break

    if date_col is None:
        return pd.DataFrame()

    df = df.rename(columns={date_col: "date"})

    required_cols = ["date", "open", "high", "low", "close", "volume"]
    keep_cols = [c for c in required_cols if c in df.columns]

    if "adj_close" in df.columns:
        keep_cols.append("adj_close")

    df = df[keep_cols].copy()

    if "adj_close" not in df.columns:
        df["adj_close"] = df["close"]

    df["ticker"] = ticker
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])

    numeric_cols = ["open", "high", "low", "close", "adj_close", "volume"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["close", "volume"])
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

    return df


def add_market_features(df):
    df = df.copy()
    df = df.sort_values(["ticker", "date"])

    df["daily_return"] = df.groupby("ticker")["close"].pct_change()
    df["log_return"] = np.log(df["close"] / df.groupby("ticker")["close"].shift(1))

    df["return_5d_forward"] = (
        df.groupby("ticker")["close"].shift(-5) / df["close"] - 1
    )

    df["return_10d_forward"] = (
        df.groupby("ticker")["close"].shift(-10) / df["close"] - 1
    )

    df["rolling_volatility_20d"] = (
        df.groupby("ticker")["daily_return"]
        .rolling(20)
        .std()
        .reset_index(level=0, drop=True)
    )

    df["volume_ma_20d"] = (
        df.groupby("ticker")["volume"]
        .rolling(20)
        .mean()
        .reset_index(level=0, drop=True)
    )

    df["volume_std_20d"] = (
        df.groupby("ticker")["volume"]
        .rolling(20)
        .std()
        .reset_index(level=0, drop=True)
    )

    df["volume_zscore"] = (
        (df["volume"] - df["volume_ma_20d"]) / df["volume_std_20d"]
    )

    df["high_5d_forward"] = (
        df.groupby("ticker")["high"]
        .rolling(5)
        .max()
        .shift(-5)
        .reset_index(level=0, drop=True)
    )

    df["low_5d_forward"] = (
        df.groupby("ticker")["low"]
        .rolling(5)
        .min()
        .shift(-5)
        .reset_index(level=0, drop=True)
    )

    df["drawdown_5d_forward"] = df["low_5d_forward"] / df["close"] - 1

    return df


def download_yfinance_data(tickers, start_date, end_date, force_download=False):
    all_rows = []

    for ticker in tickers:
        out_path = YFINANCE_DIR / f"{ticker}.csv"

        use_cache = False

        if not force_download and out_path.exists():
            cached = read_csv_if_exists(out_path)

            if cached is not None and not cached.empty and "date" in [c.lower() for c in cached.columns]:
                cached = normalize_yfinance_frame(cached, ticker)

                if not cached.empty:
                    all_rows.append(cached)
                    use_cache = True

        if use_cache:
            continue

        print(f"Downloading {ticker} from yfinance...")

        df = yf.download(
            ticker,
            start=start_date,
            end=end_date,
            auto_adjust=False,
            progress=False
        )

        df = normalize_yfinance_frame(df, ticker)

        if df.empty:
            print(f"Warning: no usable data for {ticker}")
            continue

        save_csv(df, out_path)
        all_rows.append(df)

        time.sleep(0.5)

    if not all_rows:
        raise ValueError("No market data was loaded. Check ticker symbols or internet connection.")

    market_df = pd.concat(all_rows, ignore_index=True)
    market_df = add_market_features(market_df)

    save_csv(market_df, PROCESSED_DIR / "market_features.csv")

    return market_df


# Run
market_df = download_yfinance_data(
    TICKERS,
    START_DATE,
    END_DATE,
    force_download=True
)

market_df.head()

,date,open,high,low,close,volume,adj_close,ticker,daily_return,log_return,return_5d_forward,return_10d_forward,rolling_volatility_20d,volume_ma_20d,volume_std_20d,volume_zscore,high_5d_forward,low_5d_forward,drawdown_5d_forward
3765,2022-01-03,177.830002,182.880005,177.710007,182.009995,104487900,177.939728,AAPL,NaN,NaN,-0.053953,-0.067084,NaN,NaN,NaN,NaN,182.940002,168.169998,-0.076040
3766,2022-01-04,182.630005,182.940002,179.119995,179.699997,99310400,175.681381,AAPL,-0.012692,-0.012773,-0.025709,-0.074958,NaN,NaN,NaN,NaN,180.169998,168.169998,-0.064162
3767,2022-01-05,179.610001,180.169998,174.639999,174.919998,94537600,171.008301,AAPL,-0.026600,-0.026960,0.003487,-0.059513,NaN,NaN,NaN,NaN,177.179993,168.169998,-0.038589
3768,2022-01-06,172.699997,175.300003,171.639999,172.000000,96904000,168.153580,AAPL,-0.016693,-0.016834,0.001105,-0.055756,NaN,NaN,NaN,NaN,177.179993,168.169998,-0.022267
3769,2022-01-07,172.889999,174.139999,171.029999,172.169998,86709100,168.319794,AAPL,0.000988,0.000988,0.005227,-0.061277,NaN,NaN,NaN,NaN,177.179993,168.169998,-0.023233


## 3. Collect Google Trends Data

Google Trends values are relative scores from 0 to 100, not absolute search counts.

Important interpretation:

- `100` means peak relative search popularity in the selected query/timeframe.
- `0` means very low relative interest, not necessarily zero searches.
- Weekly data starts on Sunday, so dates may begin slightly before the requested start date.

In [13]:
def download_google_trends(keywords, start_date, end_date, geo="US", force_download=False):
    out_path = TRENDS_DIR / f"google_trends_{geo}_{start_date}_{end_date}.csv"

    if not force_download:
        cached = read_csv_if_exists(out_path)
        if cached is not None:
            cached["date"] = pd.to_datetime(cached["date"])
            return cached

    if TrendReq is None:
        raise ImportError("pytrends is not installed. Run: pip install pytrends")

    if len(keywords) > 5:
        raise ValueError("Google Trends allows a maximum of 5 keywords per request.")

    pytrends = TrendReq(hl="en-US", tz=360)

    print("Downloading Google Trends data")
    pytrends.build_payload(
        keywords,
        timeframe=f"{start_date} {end_date}",
        geo=geo
    )

    df = pytrends.interest_over_time().reset_index()

    if "isPartial" in df.columns:
        df = df.drop(columns=["isPartial"])

    df = df.rename(columns={"date": "date"})
    df["date"] = pd.to_datetime(df["date"])

    save_csv(df, out_path)
    return df

trends_df = download_google_trends(AI_TREND_KEYWORDS, START_DATE, END_DATE, GEO, force_download=False)
trends_df.head()

,date,artificial intelligence,ChatGPT,Claude AI,generative AI,machine learning
0,2021-12-26,1,0,0,0,1
1,2022-01-02,1,0,0,0,1
2,2022-01-09,1,0,0,0,2
3,2022-01-16,1,0,0,0,1
4,2022-01-23,1,0,0,0,2


In [7]:
def create_trends_features(trends_df: pd.DataFrame) -> pd.DataFrame:
    df = trends_df.copy()
    keyword_cols = [c for c in df.columns if c != "date"]

    # Combined attention score
    df["ai_trend_score"] = df[keyword_cols].mean(axis=1)

    # Rolling trend features
    df["ai_trend_4w_mean"] = df["ai_trend_score"].rolling(4).mean()
    df["ai_trend_4w_change"] = df["ai_trend_score"] - df["ai_trend_4w_mean"]

    # Spike indicator based on historical distribution
    threshold = df["ai_trend_score"].quantile(0.90)
    df["ai_trend_spike"] = (df["ai_trend_score"] >= threshold).astype(int)

    df["ai_trend_bucket"] = bucket_series(df["ai_trend_score"]).astype(str)

    return df

trends_features_df = create_trends_features(trends_df)
save_csv(trends_features_df, PROCESSED_DIR / "google_trends_features.csv")
trends_features_df.tail()

,date,artificial intelligence,ChatGPT,Claude AI,generative AI,machine learning,ai_trend_score,ai_trend_4w_mean,ai_trend_4w_change,ai_trend_spike,ai_trend_bucket
153,2024-12-01,2,81,1,1,2,17.4,15.95,1.45,1,high
154,2024-12-08,2,100,1,1,2,21.2,17.10,4.10,1,high
155,2024-12-15,1,75,1,1,2,16.0,16.65,-0.65,1,high
156,2024-12-22,1,45,1,1,1,9.8,16.10,-6.30,0,high
157,2024-12-29,1,47,1,1,1,10.2,14.30,-4.10,0,high


## 4. Collect SEC EDGAR Filings and Extract AI Adoption Signals

SEC filings are large because they contain:

- Human-readable annual report text
- HTML formatting
- XBRL accounting tags
- Metadata and exhibits

For this project, we extract simple filing-level signals:

- Total word count
- AI keyword counts
- AI keyword density
- Filing date / report year
- Year-over-year change in AI density

In [14]:
import re
import time
from pathlib import Path

import pandas as pd

try:
    from bs4 import BeautifulSoup
except ImportError:
    BeautifulSoup = None

try:
    from sec_edgar_downloader import Downloader
except ImportError:
    Downloader = None


# -----------------------------
# SEC CONFIG
# -----------------------------

SEC_COMPANY_NAME = "SignalScope"
SEC_EMAIL = "rastogi.o@northeastern.edu"

# IMPORTANT:
# Use this if your downloaded SEC folder is:
# data/sec/sec-edgar-filings/...
SEC_DIR = Path("data/sec")

# Use this instead only if your downloaded SEC folder is:
# data/raw/sec/sec-edgar-filings/...
# SEC_DIR = Path("data/raw/sec")

SEC_FILINGS_DIR = SEC_DIR / "sec-edgar-filings"

SEC_AI_KEYWORDS = [
    "artificial intelligence",
    "generative ai",
    "machine learning",
    "large language model",
    "llm",
    "data center",
    "accelerated computing",
    "automation",
    "gpu",
    "cloud computing"
]


# -----------------------------
# DOWNLOAD SEC FILINGS
# -----------------------------

def download_sec_filings(
    tickers,
    form_type="10-K",
    after="2022-01-01",
    before="2025-01-01",
    force_download=False
):
    if Downloader is None:
        raise ImportError(
            "sec-edgar-downloader is not installed. Run: pip install sec-edgar-downloader"
        )

    if not SEC_EMAIL or SEC_EMAIL == "your_email@example.com":
        raise ValueError("Set SEC_EMAIL to your real email before downloading SEC filings.")

    dl = Downloader(
        company_name=SEC_COMPANY_NAME,
        email_address=SEC_EMAIL,
        download_folder=str(SEC_DIR)
    )

    for ticker in tickers:
        ticker_dir = SEC_FILINGS_DIR / ticker / form_type

        if ticker_dir.exists() and not force_download:
            existing = list(ticker_dir.glob("*/full-submission.txt"))
            if existing:
                print(f"{ticker}: already has {len(existing)} {form_type} filings. Skipping.")
                continue

        print(f"Downloading {form_type} filings for {ticker}...")

        try:
            count = dl.get(
                form_type,
                ticker,
                after=after,
                before=before
            )
            print(f"{ticker}: downloaded/found {count} filings")
            time.sleep(1)

        except Exception as e:
            print(f"Failed for {ticker}: {e}")


# -----------------------------
# TEXT EXTRACTION
# -----------------------------

def extract_text_from_sec_submission(file_path: Path) -> str:
    raw = file_path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    # Prefer the main 10-K document block.
    document_blocks = re.findall(
        r"<DOCUMENT>(.*?)</DOCUMENT>",
        raw,
        flags=re.DOTALL | re.IGNORECASE
    )

    selected_block = None

    for block in document_blocks:
        type_match = re.search(
            r"<TYPE>\s*([^\n\r<]+)",
            block,
            flags=re.IGNORECASE
        )

        if type_match:
            doc_type = type_match.group(1).strip().upper()

            if doc_type.startswith("10-K"):
                selected_block = block
                break

    if selected_block is None:
        selected_block = raw

    text_match = re.search(
        r"<TEXT>(.*?)</TEXT>",
        selected_block,
        flags=re.DOTALL | re.IGNORECASE
    )

    html = text_match.group(1) if text_match else selected_block

    if BeautifulSoup is not None:
        soup = BeautifulSoup(html, "html.parser")

        # Remove hidden XBRL/header/script/style noise
        for tag in soup(["script", "style", "ix:header", "xbrli:context", "xbrli:unit"]):
            tag.decompose()

        text = soup.get_text(" ")

    else:
        text = re.sub(r"<[^>]+>", " ", html)

    text = re.sub(r"&nbsp;", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.lower().strip()


def extract_sec_dates(file_path: Path):
    raw_head = file_path.read_text(
        encoding="utf-8",
        errors="ignore"
    )[:10000]

    filed_match = re.search(
        r"FILED AS OF DATE:\s*(\d{8})",
        raw_head
    )

    period_match = re.search(
        r"CONFORMED PERIOD OF REPORT:\s*(\d{8})",
        raw_head
    )

    filed_date = (
        pd.to_datetime(filed_match.group(1), format="%Y%m%d", errors="coerce")
        if filed_match else pd.NaT
    )

    period_of_report = (
        pd.to_datetime(period_match.group(1), format="%Y%m%d", errors="coerce")
        if period_match else pd.NaT
    )

    return filed_date, period_of_report


# -----------------------------
# FEATURE EXTRACTION
# -----------------------------

def count_keyword_occurrences(text: str, keyword: str) -> int:
    keyword = keyword.lower().strip()

    # Phrase-safe matching.
    # Example: "ai" should not match inside "said" or "chain".
    pattern = r"(?<![a-zA-Z])" + re.escape(keyword) + r"(?![a-zA-Z])"

    return len(re.findall(pattern, text, flags=re.IGNORECASE))


def safe_keyword_col(keyword: str) -> str:
    return (
        keyword.lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace(".", "")
    )


def parse_sec_filings(
    sec_root: Path,
    tickers,
    form_type="10-K",
    keywords=None
) -> pd.DataFrame:
    keywords = keywords or SEC_AI_KEYWORDS
    rows = []

    sec_root = Path(sec_root)
    filings_root = sec_root / "sec-edgar-filings"

    print(f"Looking for SEC filings inside: {filings_root.resolve()}")

    for ticker in tickers:
        ticker_dir = filings_root / ticker / form_type

        if not ticker_dir.exists():
            print(f"No SEC directory found for {ticker}: {ticker_dir.resolve()}")
            continue

        submission_files = list(ticker_dir.glob("*/full-submission.txt"))

        if not submission_files:
            print(f"No full-submission.txt files found for {ticker}: {ticker_dir.resolve()}")
            continue

        for submission_path in submission_files:
            try:
                accession = submission_path.parent.name

                filed_date, period_of_report = extract_sec_dates(submission_path)
                text = extract_text_from_sec_submission(submission_path)

                word_count = len(text.split())

                row = {
                    "ticker": ticker,
                    "form_type": form_type,
                    "accession": accession,
                    "filed_date": filed_date,
                    "period_of_report": period_of_report,
                    "file_path": str(submission_path),
                    "word_count": word_count,
                }

                total_count = 0

                for kw in keywords:
                    col = safe_keyword_col(kw)
                    count = count_keyword_occurrences(text, kw)

                    row[f"{col}_count"] = count
                    row[f"{col}_density"] = count / max(word_count, 1)

                    total_count += count

                row["sec_ai_total_count"] = total_count
                row["sec_ai_density"] = total_count / max(word_count, 1)

                rows.append(row)

            except Exception as e:
                print(f"Failed to parse {submission_path}: {e}")

    sec_features = pd.DataFrame(rows)

    if sec_features.empty:
        print("No SEC features created. Check SEC_DIR and whether filings were downloaded.")
        return sec_features

    sec_features = sec_features.sort_values(
        ["ticker", "period_of_report", "filed_date"]
    ).reset_index(drop=True)

    sec_features["sec_ai_density_yoy_change"] = (
        sec_features
        .groupby("ticker")["sec_ai_density"]
        .diff()
    )

    sec_features["sec_ai_count_yoy_change"] = (
        sec_features
        .groupby("ticker")["sec_ai_total_count"]
        .diff()
    )

    out_path = PROCESSED_DIR / "sec_filing_features.csv"
    save_csv(sec_features, out_path)

    print(f"Saved SEC features to: {out_path.resolve()}")

    return sec_features


# -----------------------------
# RUN
# -----------------------------

# Step 1: Run this once to download filings.
download_sec_filings(
    TICKERS,
    form_type="10-K",
    after=START_DATE,
    before=END_DATE,
    force_download=False
)

# Step 2: Parse filings.
sec_features_df = parse_sec_filings(
    SEC_DIR,
    TICKERS,
    form_type="10-K",
    keywords=SEC_AI_KEYWORDS
)

sec_features_df.head()

NVDA: already has 3 10-K filings. Skipping.
MSFT: downloaded/found 3 filings
GOOG: downloaded/found 3 filings
META: downloaded/found 3 filings
AMZN: downloaded/found 3 filings
AAPL: downloaded/found 3 filings
Looking for SEC filings inside: E:\Om\signalscope\data\sec\sec-edgar-filings
Saved SEC features to: E:\Om\signalscope\data\processed\sec_filing_features.csv


,ticker,form_type,accession,filed_date,period_of_report,file_path,word_count,artificial_intelligence_count,artificial_intelligence_density,generative_ai_count,...,automation_count,automation_density,gpu_count,gpu_density,cloud_computing_count,cloud_computing_density,sec_ai_total_count,sec_ai_density,sec_ai_density_yoy_change,sec_ai_count_yoy_change
0,AAPL,10-K,0000320193-22-000108,2022-10-28,2022-09-24,data\sec\sec-edgar-filings\AAPL\10-K\000032019...,32872,0,0.000000,0,...,0,0.0,0,0.0,0,0.0,3,0.000091,NaN,NaN
1,AAPL,10-K,0000320193-23-000106,2023-11-03,2023-09-30,data\sec\sec-edgar-filings\AAPL\10-K\000032019...,30650,1,0.000033,0,...,0,0.0,0,0.0,0,0.0,5,0.000163,0.000072,2.0
2,AAPL,10-K,0000320193-24-000123,2024-11-01,2024-09-28,data\sec\sec-edgar-filings\AAPL\10-K\000032019...,31122,3,0.000096,0,...,0,0.0,0,0.0,0,0.0,8,0.000257,0.000094,3.0
3,AMZN,10-K,0001018724-22-000005,2022-02-04,2021-12-31,data\sec\sec-edgar-filings\AMZN\10-K\000101872...,38518,3,0.000078,0,...,0,0.0,0,0.0,0,0.0,15,0.000389,NaN,NaN
4,AMZN,10-K,0001018724-23-000004,2023-02-03,2022-12-31,data\sec\sec-edgar-filings\AMZN\10-K\000101872...,40772,3,0.000074,0,...,0,0.0,0,0.0,0,0.0,16,0.000392,0.000003,1.0


## 5. Optional Finnhub News Data

Finnhub is useful for company-specific news headlines and summaries.

Use it as a supplementary signal:

- `ai_news_count`
- `ai_news_spike`

Do not rely on it as the main historical source because the free plan has API and historical-data limits.

In [ ]:
FINNHUB_API_KEY = os.getenv("FINNHUB_API_KEY", "test_fc237ef0fc71a10d050b9d1b6d7b96f9ca7d56aaa272bf738d223d689e0")

def download_finnhub_news(tickers, start_date, end_date, force_download=False):
    if finnhub is None:
        print("finnhub-python is not installed. Run: pip install finnhub-python")
        return pd.DataFrame()

    if not FINNHUB_API_KEY:
        print("No FINNHUB_API_KEY found. Set it as an environment variable. Skipping Finnhub.")
        return pd.DataFrame()

    client = finnhub.Client(api_key=FINNHUB_API_KEY)
    all_rows = []

    for ticker in tickers:
        out_path = FINNHUB_DIR / f"{ticker}_news_{start_date}_{end_date}.csv"

        if not force_download:
            cached = read_csv_if_exists(out_path)
            if cached is not None:
                all_rows.append(cached)
                continue

        print(f"Downloading Finnhub news for {ticker}")
        try:
            news = client.company_news(ticker, _from=start_date, to=end_date)
            df = pd.DataFrame(news)

            if len(df) > 0:
                df["ticker"] = ticker
                df["date"] = pd.to_datetime(df["datetime"], unit="s").dt.date
                save_csv(df, out_path)
                all_rows.append(df)

            time.sleep(1)
        except Exception as e:
            print(f"Failed for {ticker}: {e}")

    if not all_rows:
        return pd.DataFrame()

    news_df = pd.concat(all_rows, ignore_index=True)
    return news_df

news_df = download_finnhub_news(TICKERS, "2024-01-01", "2024-12-31", force_download=False)
news_df.head() if len(news_df) else news_df

In [ ]:
def create_news_features(news_df: pd.DataFrame, keywords=None) -> pd.DataFrame:
    if news_df is None or len(news_df) == 0:
        return pd.DataFrame(columns=["date", "ticker", "ai_news_count"])

    keywords = keywords or SEC_AI_KEYWORDS

    df = news_df.copy()
    df["date"] = pd.to_datetime(df["date"])

    def is_ai_related(row):
        text = f"{row.get('headline', '')} {row.get('summary', '')}".lower()
        return any(kw.lower() in text for kw in keywords)

    df["is_ai_related"] = df.apply(is_ai_related, axis=1)

    daily = (
        df[df["is_ai_related"]]
        .groupby(["date", "ticker"])
        .size()
        .reset_index(name="ai_news_count")
    )

    if len(daily) > 0:
        daily["ai_news_spike"] = (
            daily.groupby("ticker")["ai_news_count"]
            .transform(lambda s: (s >= s.quantile(0.90)).astype(int))
        )

    save_csv(daily, PROCESSED_DIR / "finnhub_news_features.csv")
    return daily

news_features_df = create_news_features(news_df)
news_features_df.head()

## 6. Build the Final EDA Dataset

The final dataset combines:

- Daily stock data
- Weekly Google Trends data, forward-filled to trading days
- Filing-level SEC features, merged as-of by filing/report date
- Optional daily news features

In [ ]:
def merge_features(market_df, trends_features_df, sec_features_df=None, news_features_df=None):
    df = market_df.copy()
    df["date"] = pd.to_datetime(df["date"])

    # Google Trends is weekly. Merge as-of so every trading day gets most recent trend value.
    trends = trends_features_df.copy()
    trends["date"] = pd.to_datetime(trends["date"])
    trends = trends.sort_values("date")

    df = df.sort_values("date")
    df = pd.merge_asof(
        df,
        trends,
        on="date",
        direction="backward"
    )

    # SEC filings are sparse. Merge most recent filing signal by ticker.
    if sec_features_df is not None and len(sec_features_df) > 0:
        sec = sec_features_df.copy()
        sec["sec_signal_date"] = sec["filed_date"].fillna(sec["period_of_report"])
        sec = sec.dropna(subset=["sec_signal_date"])
        sec = sec.sort_values(["ticker", "sec_signal_date"])

        merged_rows = []
        for ticker, g in df.groupby("ticker"):
            g = g.sort_values("date")
            sec_g = sec[sec["ticker"] == ticker].sort_values("sec_signal_date")

            if len(sec_g) == 0:
                merged_rows.append(g)
                continue

            g2 = pd.merge_asof(
                g,
                sec_g.drop(columns=["ticker"]),
                left_on="date",
                right_on="sec_signal_date",
                direction="backward"
            )
            merged_rows.append(g2)

        df = pd.concat(merged_rows, ignore_index=True)

    # News features are daily.
    if news_features_df is not None and len(news_features_df) > 0:
        news = news_features_df.copy()
        news["date"] = pd.to_datetime(news["date"])

        df = df.merge(news, on=["date", "ticker"], how="left")
        df["ai_news_count"] = df["ai_news_count"].fillna(0)
        if "ai_news_spike" in df.columns:
            df["ai_news_spike"] = df["ai_news_spike"].fillna(0)

    # Buckets for future RL state space
    if "volume_zscore" in df.columns:
        df["volume_bucket"] = df.groupby("ticker")["volume_zscore"].transform(lambda s: bucket_series(s).astype(str))

    if "daily_return" in df.columns:
        df["recent_return_bucket"] = df.groupby("ticker")["daily_return"].transform(lambda s: bucket_series(s).astype(str))

    if "volatility_10d" in df.columns:
        df["volatility_bucket"] = df.groupby("ticker")["volatility_10d"].transform(lambda s: bucket_series(s).astype(str))

    save_csv(df, PROCESSED_DIR / "signalscope_eda_dataset.csv")
    return df

eda_df = merge_features(
    market_df,
    trends_features_df,
    sec_features_df=sec_features_df,
    news_features_df=news_features_df
)

eda_df.head()

## 7. Dataset Quality Checks

Before plotting, verify:

- Number of rows
- Date range
- Missing values
- Per-ticker coverage
- Feature availability

In [ ]:
print("Shape:", eda_df.shape)
print("\nDate range:", eda_df["date"].min(), "to", eda_df["date"].max())
print("\nTickers:", sorted(eda_df["ticker"].dropna().unique()))

display(eda_df.groupby("ticker").agg(
    start_date=("date", "min"),
    end_date=("date", "max"),
    rows=("date", "count"),
    avg_return=("daily_return", "mean"),
    avg_volume=("volume", "mean")
))

missing = (
    eda_df.isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing.columns = ["column", "missing_fraction"]
display(missing.head(30))

## 8. EDA Plotting Helpers

In [ ]:
def plot_line(df, x, y, title, ticker=None):
    temp = df.copy()
    if ticker is not None:
        temp = temp[temp["ticker"] == ticker]

    plt.figure(figsize=(12, 5))
    plt.plot(temp[x], temp[y])
    plt.title(title)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_hist(df, column, title, bins=50):
    plt.figure(figsize=(10, 5))
    plt.hist(df[column].dropna(), bins=bins)
    plt.title(title)
    plt.xlabel(column)
    plt.ylabel("Frequency")
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_scatter(df, x, y, title):
    plt.figure(figsize=(8, 5))
    plt.scatter(df[x], df[y], alpha=0.4)
    plt.title(title)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_bar(series, title, xlabel="", ylabel=""):
    plt.figure(figsize=(10, 5))
    series.plot(kind="bar")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, axis="y", alpha=0.3)
    plt.show()

## 9. Market Data EDA

Start with price, returns, volume, volatility, and drawdown behavior.

In [ ]:
for ticker in TICKERS:
    plot_line(eda_df, "date", "close", f"{ticker}: Closing Price", ticker=ticker)

In [ ]:
plot_hist(eda_df, "daily_return", "Distribution of Daily Returns")
plot_hist(eda_df, "return_5d_fwd", "Distribution of 5-Day Forward Returns")
plot_hist(eda_df, "drawdown_5d_fwd", "Distribution of 5-Day Forward Drawdowns")

In [ ]:
summary_by_ticker = eda_df.groupby("ticker").agg(
    mean_daily_return=("daily_return", "mean"),
    std_daily_return=("daily_return", "std"),
    mean_5d_forward_return=("return_5d_fwd", "mean"),
    worst_5d_drawdown=("drawdown_5d_fwd", "min"),
    avg_volume_zscore=("volume_zscore", "mean")
).sort_values("mean_5d_forward_return", ascending=False)

display(summary_by_ticker)

## 10. Google Trends EDA

Here we check whether retail attention spikes are visible and whether they line up with market behavior.

In [ ]:
keyword_cols = [c for c in trends_df.columns if c != "date"]

plt.figure(figsize=(12, 5))
for col in keyword_cols:
    plt.plot(trends_df["date"], trends_df[col], label=col)

plt.title("Google Trends: AI-Related Search Interest")
plt.xlabel("Date")
plt.ylabel("Relative Search Interest")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plot_line(trends_features_df, "date", "ai_trend_score", "Combined AI Trend Score")
plot_hist(trends_features_df, "ai_trend_score", "Distribution of AI Trend Score")

In [ ]:
trend_spike_dates = trends_features_df[trends_features_df["ai_trend_spike"] == 1][["date", "ai_trend_score"]]
display(trend_spike_dates.head(20))
print("Number of trend spike weeks:", len(trend_spike_dates))

## 11. Market Behavior Around AI Trend Spikes

This checks whether high Google Trends periods are associated with different 5-day forward returns, volatility, or drawdowns.

In [ ]:
display(
    eda_df.groupby("ai_trend_bucket").agg(
        rows=("date", "count"),
        avg_5d_return=("return_5d_fwd", "mean"),
        median_5d_return=("return_5d_fwd", "median"),
        avg_10d_volatility=("volatility_10d", "mean"),
        avg_5d_drawdown=("drawdown_5d_fwd", "mean"),
        worst_5d_drawdown=("drawdown_5d_fwd", "min"),
    )
)

In [ ]:
spike_comparison = eda_df.groupby("ai_trend_spike").agg(
    rows=("date", "count"),
    avg_5d_return=("return_5d_fwd", "mean"),
    median_5d_return=("return_5d_fwd", "median"),
    avg_volume_zscore=("volume_zscore", "mean"),
    avg_volatility_10d=("volatility_10d", "mean"),
    avg_drawdown_5d=("drawdown_5d_fwd", "mean"),
)
display(spike_comparison)

In [ ]:
plot_scatter(
    eda_df.dropna(subset=["ai_trend_score", "return_5d_fwd"]),
    "ai_trend_score",
    "return_5d_fwd",
    "AI Trend Score vs 5-Day Forward Return"
)

plot_scatter(
    eda_df.dropna(subset=["ai_trend_score", "volume_zscore"]),
    "ai_trend_score",
    "volume_zscore",
    "AI Trend Score vs Volume Z-Score"
)

## 12. SEC Filing Signal EDA

This section only works after SEC filings have been downloaded and parsed.

The most important SEC feature for v1 is:

```text
sec_ai_density = AI keyword mentions / total filing words
```

This avoids unfairly comparing long filings with short filings.

In [ ]:
if sec_features_df is not None and len(sec_features_df) > 0:
    display(sec_features_df[[
        "ticker", "accession", "period_of_report", "filed_date",
        "word_count", "sec_ai_total_count", "sec_ai_density",
        "sec_ai_density_yoy_change"
    ]].sort_values(["ticker", "period_of_report"]))
else:
    print("No SEC features found yet. Download SEC filings first.")

In [ ]:
if sec_features_df is not None and len(sec_features_df) > 0:
    for ticker in sec_features_df["ticker"].unique():
        temp = sec_features_df[sec_features_df["ticker"] == ticker].sort_values("period_of_report")
        plt.figure(figsize=(8, 4))
        plt.plot(temp["period_of_report"], temp["sec_ai_density"], marker="o")
        plt.title(f"{ticker}: SEC AI Mention Density Over Time")
        plt.xlabel("Report Period")
        plt.ylabel("AI Mention Density")
        plt.grid(True, alpha=0.3)
        plt.show()
else:
    print("No SEC features available for plotting.")

## 13. Optional News EDA

If Finnhub news was collected, this checks whether company-specific AI news spikes are associated with returns or volume.

In [ ]:
if news_features_df is not None and len(news_features_df) > 0:
    display(news_features_df.head())

    display(
        eda_df.groupby("ticker").agg(
            total_ai_news=("ai_news_count", "sum"),
            avg_ai_news=("ai_news_count", "mean"),
            avg_5d_return=("return_5d_fwd", "mean")
        ).sort_values("total_ai_news", ascending=False)
    )
else:
    print("No Finnhub news features found. This is optional.")

## 14. Correlation Analysis

Correlations do not prove causality, but they help identify whether a feature is worth using in the RL state space.

In [ ]:
candidate_corr_cols = [
    "daily_return",
    "return_5d_fwd",
    "return_10d_fwd",
    "drawdown_5d_fwd",
    "volume_zscore",
    "volatility_10d",
    "ai_trend_score",
    "ai_trend_4w_change",
    "sec_ai_density",
    "sec_ai_density_yoy_change",
    "ai_news_count",
]

corr_cols = [c for c in candidate_corr_cols if c in eda_df.columns]

corr = eda_df[corr_cols].corr()

display(corr)

plt.figure(figsize=(9, 7))
plt.imshow(corr, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Correlation Matrix")
plt.colorbar()
plt.tight_layout()
plt.show()

## 15. Candidate RL State Features

After EDA, we do not feed raw continuous values directly into tabular Q-learning.  
We bucket them into discrete states.

Example state:

```text
ai_trend_bucket = high
volume_bucket = medium
recent_return_bucket = low
volatility_bucket = high
```

In [ ]:
state_cols = [
    "ai_trend_bucket",
    "volume_bucket",
    "recent_return_bucket",
    "volatility_bucket",
]

available_state_cols = [c for c in state_cols if c in eda_df.columns]

eda_df["rl_state"] = eda_df[available_state_cols].astype(str).agg("|".join, axis=1)

display(eda_df[["date", "ticker"] + available_state_cols + ["rl_state", "return_5d_fwd", "drawdown_5d_fwd"]].head(20))

state_counts = eda_df["rl_state"].value_counts().head(20)
plot_bar(state_counts, "Top 20 Most Common RL States", xlabel="State", ylabel="Count")

## 16. EDA Findings Template

Fill this section after running the notebook.

### Data coverage

- Number of tickers:
- Date range:
- Market rows:
- Google Trends rows:
- SEC filings parsed:
- News rows, if used:

### Early observations

1. Which AI keyword dominates Google Trends?
2. Do AI trend spikes align with higher volume?
3. Are 5-day forward returns better, worse, or unchanged after AI trend spikes?
4. Which ticker has the strongest reaction?
5. Are SEC AI mentions increasing over time?
6. Is the signal strong enough for a Q-learning prototype?

### Recommended v1 RL features

Use only 3 to 4 state features initially:

- `ai_trend_bucket`
- `volume_bucket`
- `recent_return_bucket`
- `volatility_bucket`

Keep SEC and Finnhub as analysis features first. Add them to RL only if they show meaningful variation.

## 17. Save Final Dataset

This file is the direct input for the future Q-learning environment.

In [ ]:
final_path = PROCESSED_DIR / "signalscope_eda_dataset.csv"
eda_df.to_csv(final_path, index=False)

print("Final EDA dataset saved to:")
print(final_path)

print("\nShape:", eda_df.shape)
display(eda_df.head())